# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.


## Подешавање

Пре покретања овог нотебоока, уверите се да имате:

1. **Пројекат у Microsoft Foundry** са распоређеним моделом за разговор (нпр. `gpt-5-mini`).
2. **Пријављени сте преко Azure CLI** — покрените `az login` у вашем терминалу.
3. **Постављене потребне променљиве окружења:**
   - `AZURE_AI_PROJECT_ENDPOINT` — крајња тачка вашег Microsoft Foundry пројекта.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег распоређеног модела.

Доња ћелија инсталира потребне Python пакете.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Креирање Вашег Првог Агента

Агенту су потребне две ствари:

- **Упутства** која му говоре *ко је* и *како да се понаша* (системски упит).
- **Алатке** — Python функције украшене са `@tool` које агент може позвати да би преузео информације или извршио акције.

Испод дефинишемо једноставну алатку која враћа листу популарних дестинација за одмор. Агент ће користити ову алатку када корисник тражи препоруке за путовања.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Стриминг одговора

За интерактивније корисничко искуство можете **стримовати** одговор агента. Уместо да чекате комплетан одговор, агент испоручује делове текста како се генеришу. Ово је посебно корисно у интерфејсима четова где желите да прикажете резултат у реалном времену.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

- **Креирате провајдера** који се повезује са Microsoft Foundry Agent Service преко `FoundryChatClient`.
- **Дефинишете алат** користећи `@tool` декоратор тако да агент може да позове ваше Python функције.
- **Покренете агента** са поруком корисника и одштампате његов одговор.
- **Стримујете одговоре** за излаз у реалном времену.

У следећој лекцији ћемо детаљније истражити агенцки оквир и научити како да агентима дамо моћније алатке и способности више-степеног расуђивања.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
